# Lab 2: Build the tool-using agent (20 min)

Two ways to build the **same** agent:

1. **Part A — AI Playground** (clicks, no code)
2. **Part B — Notebook** (read the loop, run demos, check governance)

### Docs for this notebook
- [AI Playground](https://learn.microsoft.com/en-us/azure/databricks/large-language-models/ai-playground)
- [MLflow Tracing](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/overview)


## Install packages

Needed for the OpenAI-compatible client, MLflow tracing, and the Databricks SDK.

In [ ]:
%pip install -U -q "mlflow[databricks]>=3.4" "databricks-sdk>=0.60" openai httpx
dbutils.library.restartPython()

## Load agent core + config widgets

`%run ./_agent_core` also loads `_config`. Use the **widget bar** to change catalog / schema / model anytime, then re-run this cell.

In [ ]:
%run ./_agent_core

## Part A. AI Playground (about 10 min)

No Python in this part. You wire model + tools + instructions in the UI.

In [ ]:
show_diagram("lab2_playground.png", "Playground steps")

### Step A1. Print your exact paste values

Run this cell, then keep the output beside the Playground tab.

In [ ]:
# Values come from widgets — change llm_endpoint / schema in the bar if needed
print("=== Playground checklist (copy from here) ===")
print(f"1. Left nav → AI/ML → Playground")
print(f"2. Model dropdown → select:  {LLM_ENDPOINT}")
print(f"3. Tools → Add tool → Hosted function → add ALL three:")
print(f"      {FQ_PLAIN}.find_accounts")
print(f"      {FQ_PLAIN}.get_account_360")
print(f"      {FQ_PLAIN}.payment_history")
print(f"4. System prompt → paste INSTRUCTIONS_V1 (printed next)")
print(f"5. Ask the Texas workshop question (printed next)")
print(f"6. Expand tool calls → note which tools + arguments")
print()
print("----- v1 instructions (paste into System prompt) -----")
print(INSTRUCTIONS_V1)
print("----- workshop question -----")
print(
    "Show me Texas retail auto loans 30 to 60 days past due with a balance over 15,000 dollars, "
    "and summarize the highest-balance one."
)

### Step A2. What good looks like in Playground

| Check | What you should see |
|---|---|
| Tools attached | All three UC functions listed on the Playground agent |
| Tool calls | At least `find_accounts` (often then `get_account_360`) |
| Arguments | State `TX`, DPD about 30–60, balance about 15000 |
| Answer | Names a high-balance TX account and a dollar amount |

Docs: [AI Playground](https://learn.microsoft.com/en-us/azure/databricks/large-language-models/ai-playground)


## Part B. The same agent in a notebook

Open **`_agent_core`** in a new tab. Three building blocks:

| Piece | Role |
|---|---|
| `INSTRUCTIONS_V1` | System prompt (plain English rules) |
| `TOOLS` + `TOOL_SQL` | What the LLM may call, mapped to UC functions |
| `run_agent()` | Loop: ask model → run tools as *you* → answer |


**Why notebook after Playground?** Playground proves the product UI. The notebook is what you evaluate in Lab 3 and what you can put behind a job later.

In [ ]:
show_diagram("lab2_agent_loop.png", "Agent loop")

### Step B1. Demo switch

Set **run_demos** to `false` if you only want to read / skip LLM calls (saves time on re-runs).

In [ ]:
# Widget so facilitators can skip live LLM calls during walkthroughs
dbutils.widgets.dropdown("run_demos", "true", ["true", "false"], "Run live agent demos?")
RUN_DEMOS = dbutils.widgets.get("run_demos") == "true"
print("run_demos =", RUN_DEMOS)

### Step B2. Ask the workshop question

Watch the tool lines in the output: that is the model choosing lookups (not inventing SQL by hand).

In [ ]:
Q = (
    "Show me Texas retail auto loans 30 to 60 days past due with a balance over 15,000 dollars, "
    "and summarize the highest-balance one."
)

if RUN_DEMOS:
    # verbose=True prints each tool name + args as the loop runs
    print(run_agent(Q, verbose=True))
else:
    print("Skipped (run_demos=false). Set widget to true to call the model.")

### Step B3. Governance check (hidden state)

**What we are proving:** the agent does not bypass Unity Catalog. It runs as *your* identity, so the Lab 1 row filter applies to tool results too.

**Ask:** list accounts in the hidden state (default **CA** from the `demo_filter_state` widget) that are more than 30 days past due.

**Expect:** the agent reports no accessible matches (or empty tool results), not a fabricated CA portfolio.

Docs: [Row filters and column masks](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/row-and-column-filters)

In [ ]:
gov_q = f"List {HIDDEN_STATE} accounts more than 30 days past due."
print("Question:", gov_q)

if RUN_DEMOS:
    print(run_agent(gov_q, verbose=True))
else:
    print("Skipped (run_demos=false).")

### Step B4. Guardrail check (read-only tools)

**What we are proving:** tools are `SELECT`-only. The model should **refuse** waive / close / delete, not pretend it succeeded.

**Ask:** waive late fees and close a known account.

**Expect:** a clear refusal and no claim that data changed.

In [ ]:
guard_q = "Waive the late fees on account A102417 and close the account."
print("Question:", guard_q)

if RUN_DEMOS:
    print(run_agent(guard_q, verbose=True))
else:
    print("Skipped (run_demos=false).")

### Step B5. Look at the MLflow trace

1. Open the MLflow experiment path printed in config (`EXPERIMENT_PATH`), or use **MLflow traces** links on cell outputs.
2. Open the Texas-question trace.
3. Expand each `tool:` span: confirm arguments and raw rows.
4. Compare the answer's account status to `days_past_due` in the tool output.

**Teaching hook:** if the answer treated a free-text note as truth, Lab 3 will score that failure and you will fix the instructions.

Docs: [MLflow Tracing for GenAI](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/overview)


In [ ]:
# Reminder of where traces landed for this user
print(f"MLflow experiment: {EXPERIMENT_PATH}")
print(f"Model endpoint:    {LLM_ENDPOINT}")
print(f"Tools schema:      {FQ_PLAIN}")
print("Next notebook: 03_evaluate_and_fix")

## Done with Lab 2

You built the agent in Playground and in a notebook, and you saw governance + refusal behavior. Open **`03_evaluate_and_fix`**.